# T27 / E08 — Bỏ bằng chứng đi rồi đo lại chú ý

**Phép kiểm can thiệp duy nhất của cả đề tài.**

Mọi thí nghiệm đến giờ đều là **tương quan**: đo một tín hiệu, so với nhãn người khác gán, báo
mức khớp. Khi hai bên khớp nhau thì kết luận "cơ chế hoạt động" là một **suy luận**, không phải
một quan sát. E06 gần trực tiếp nhất, nhưng nó vẫn chỉ *quan sát* chú ý rơi ở đâu.

Cái này **can thiệp**. Mỗi claim được ghép cặp với chính nó: cùng phản hồi, đọc hai lần trên hai
ngữ cảnh mười câu **khác nhau đúng một câu ở đúng một vị trí**. Một bên có câu bằng chứng vàng,
bên kia thay bằng một câu nhiễu. Không gì khác thay đổi — không độ dài, không số đoạn, không thứ
tự, không nhãn nào do người gán.

Nếu tín hiệu chunk-aware đúng như đề tài nói, bỏ bằng chứng đi phải làm phân bố chú ý **tản ra**:
không còn gì trong ngữ cảnh để tập trung vào. Nếu nó không nhúc nhích, thì thứ đặc trưng hình
dạng bắt được trên các bộ khác **không phải** "mô hình đã tìm thấy bằng chứng".

## Nhãn nửa 'absent' không do ai gán

Rút câu vàng ra thì phản hồi khẳng định điều ngữ cảnh không chứa — đó là định nghĩa của ảo giác
ngoại lai. Nhãn `extrinsic` ở nửa ấy là **sự thật về cách dựng**, không phải một phán đoán ai đó
có thể bất đồng. Đây là điều làm E08 khác mọi thí nghiệm còn lại: T13 đã đo rằng ranh giới nội
tại–ngoại lai đánh bại hai người gán nhãn và cả Gemini, và ở đây ranh giới ấy được **dựng ra**
chứ không được **đoán**.

## Vì sao là ViWikiFC

Bộ duy nhất mà nhãn NEI **có bằng chứng** (100 % nguyên văn, xác nhận ở T11), và bộ duy nhất đủ
nhỏ — 3.814 câu từ 73 bài Wikipedia — để làm kho truy xuất cho chính nó. Mục 8 `docs/DATA.md` dự
liệu việc này từ đầu, và T16 đã dựng kho.

## Đã đo trên CPU trước khi đặt lịch GPU

```
  bằng chứng vàng có trong kho    2.090/2.090  (100 %)
  ngữ cảnh top-10                 496 token TB, 9,8 đoạn sau khi chia theo câu
  hai vế CÙNG số đoạn             87,2 %
  và khác ĐÚNG một đoạn          99,7 % trong số đó  → chỉ giữ những cặp này
  trong đó khác đúng một đoạn     100 %
  dựng được                       1.836 cặp = 3.672 dòng
```

**Một confound suýt lọt.** BM25 để câu vàng ở **hạng 0 với 94 % claim** — vị trí tương đối trung
bình 0,040. Nếu ghép theo thứ tự truy xuất thì một bộ đoán *"luôn chọn đoạn đầu"* đã thắng sàn,
và hit@1 cao sẽ chẳng chứng minh được gì. **Xáo thứ tự** mười câu trước khi ghép đưa vị trí vàng
về **0,513**, rải đều mười hạng. Cả hai vế của một cặp dùng **cùng một hoán vị**, nếu không thì
chúng khác nhau ở mười chỗ chứ không phải một.

## Hướng dự đoán, ghi trước khi thấy số

| đặc trưng | bỏ vàng đi thì phải | vì sao |
|---|---|---|
| `chunk_entropy` | **tăng** | không còn gì để tập trung |
| `chunk_max_share` | **giảm** | không đoạn nào còn trội hẳn |
| `chunk_gini` | **giảm** | các đoạn đều nhau hơn |
| `top1_top2_gap` | **giảm** | đoạn tốt nhất hết nổi bật |
| `chunk_drift` | *không dự đoán* | drift nói về chuyển động theo token |

Bảng này nằm trong `run_extrinsic.py` dưới dạng `EXPECTED_DIRECTION`, có ca kiểm thử khóa lại, để
một kết quả đi ngược **không thể** được mô tả lại thành xác nhận sau khi đã thấy số.

**Kết quả đi ngược là kết quả bác cách diễn giải của đề tài**, và phải báo cáo đúng như vậy.

## Chuẩn bị

Ô 1 giống notebook T26. Ô 2 **khác**: E08 cần `rank-bm25` cho chỉ mục BM25, mà `pip install --no-deps` cố ý không kéo phụ thuộc nào về.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: 008bd48 T27: trả lại ô chuẩn bị dữ liệu và cho tiền kiểm đi hết chuỗi (#68)


In [2]:
# Ô 2 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit của mô hình đọc 7B.
#
# rank-bm25 là thứ T27 phải thêm. Nó có trong pyproject.toml, nhưng `pip install --no-deps -e .`
# cố ý không kéo phụ thuộc nào về, nên pip vẫn in "requires rank-bm25, which is not installed"
# ở MỌI lượt chạy từ T23. Cảnh báo ấy vô hại cho T23-T26 và KHÔNG vô hại cho T27: E08 dựng chỉ
# mục BM25 nên hỏng ngay ở giây thứ 43.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate bitsandbytes rank-bm25

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.


In [3]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
#
# Ô này hỏng ba lần liên tiếp theo ba kiểu khác nhau, nên nó được viết lại để đi HẾT CHUỖI phụ
# thuộc thay vì kiểm một thứ:
#
#   T26  ô cổng chạy SAU 10 giờ GPU        -> vô dụng, dù nó báo đúng
#   T27  lần 1: chỉ kiểm shard             -> thiếu rank-bm25, hỏng ở giây 43
#   T27  lần 2: kiểm shard + goi           -> thiếu data/interim, hỏng ở giây 44
#
# Bài học chung: một ô cổng chỉ chặn được thứ nó biết phải kiểm, và vá đúng cái vừa hỏng thì lần
# sau hỏng ở loại khác. Nên ở đây phân đôi rành mạch:
#
#   PHẢI CÓ SẴN  - phiên này không tạo được: dữ liệu thô đã mount, goi da cai
#   TỰ TẠO       - các ô sau sinh ra theo thứ tự, chỉ liệt kê để đọc và đối chiếu
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config
from vihallulens.data.paths import find_raw_dir

cfg = load_config("configs/e08_extrinsic_viwikifc.yaml")
run = extraction_hash(cfg)
problems = []

# --- PHẢI CÓ SẴN -------------------------------------------------------------------------------
packages = ("rank_bm25", "torch", "transformers", "bitsandbytes", "pandas")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
status = f"THIEU {absent}" if absent else f"du ca {list(packages)}"
print(f"  goi phai co san   : {status}")
if absent:
    problems.append(f"thieu goi {absent}")

try:
    raw = find_raw_dir()
    files = sorted(p.name for p in Path(raw).glob("viwikifc*"))
    print(f"  du lieu tho       : {raw}")
    print(f"  file viwikifc tho : {files or 'KHONG CO'}")
    if not files:
        problems.append("khong thay file viwikifc nao trong du lieu tho")
except Exception as error:
    print(f"  du lieu tho       : KHONG TIM THAY ({error})")
    problems.append("chua mount dataset du lieu tho")

# --- TỰ TẠO, theo thứ tự các ô sau ---------------------------------------------------------------
chain = [
    ("o 4", "data/interim/viwikifc_dev.parquet", "normalize_data + split_data"),
    ("o 5", "data/interim/viwikifc_evidence_corpus.parquet", "build_evidence_corpus"),
    ("o 5", "data/interim/viwikifc_e08_dev.parquet", "build_retrieval_contexts"),
    ("o 7", f"data/processed/viwikifc_e08_dev_{run}.jsonl", "extract_features"),
]
print(f"  hash trich        : {run}")
print("  chuoi tu tao:")
for cell, target, maker in chain:
    print(f"    {cell:<5} {target:<48} <- {maker}")

if problems:
    raise SystemExit("TIEN KIEM HONG: " + "; ".join(problems))
print("\nTien kiem dat: dieu kien ngoai da du, phan con lai phien nay tu tao.")

  goi phai co san   : du ca ['rank_bm25', 'torch', 'transformers', 'bitsandbytes', 'pandas']
  du lieu tho       : /kaggle/input/datasets/unicorn1209/vihallulens
  file viwikifc tho : ['viwikifc_dev.csv', 'viwikifc_test.csv', 'viwikifc_train.csv']
  hash trich        : 16957fb67493
  chuoi tu tao:
    o 4   data/interim/viwikifc_dev.parquet                <- normalize_data + split_data
    o 5   data/interim/viwikifc_evidence_corpus.parquet    <- build_evidence_corpus
    o 5   data/interim/viwikifc_e08_dev.parquet            <- build_retrieval_contexts
    o 7   data/processed/viwikifc_e08_dev_16957fb67493.jsonl <- extract_features

Tien kiem dat: dieu kien ngoai da du, phan con lai phien nay tu tao.


In [4]:
# Ô 4 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
# Ô này bị bỏ sót khi dựng notebook T27 và là nguyên nhân lượt chạy thứ hai hỏng: không có
# data/interim/viwikifc_*.parquet thì kho truy xuất không dựng được.
get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset viwikifc")
get_ipython().system("python scripts/split_data.py --only viwikifc")
get_ipython().system("python -m pytest tests/test_paired_contexts.py tests/test_stats.py -q")


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : 008bd48 T27: trả lại ô chuẩn bị dữ liệu và cho tiền kiểm đi hết chuỗi (#68)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.16.1
  bitsandbytes     : 0.50.2
  accelerate       : 1.14.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIWIKIFC
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 20,919
  ngữ

In [5]:
# Ô 5 — dựng kho truy xuất và ngữ cảnh theo cặp. Khoảng 2 phút, CPU.
# Không cần GPU: chỉ là BM25 trên 3.814 câu rồi ghép văn bản.
!python scripts/build_evidence_corpus.py
!python scripts/build_retrieval_contexts.py --split dev


T16 — KHO TRUY XUẤT VIWIKIFC
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  câu bằng chứng        : 3,814
  bài Wikipedia         : 73
  độ dài câu (từ)       : trung vị 31, trung bình 35.2, dài nhất 251
  claim mỗi câu         : trung bình 5.5, tối đa 42
  đã ghi                : data/interim/viwikifc_evidence_corpus.parquet
  dựng chỉ mục BM25     : 0.08 s

--------------------------------------------------------------------------------
TRUY VẤN THỬ
--------------------------------------------------------------------------------

  claim : Thác nước Jog nằm trên sông Hằng.
  nhãn  : intrinsic  ·  bài: châu Á
    1. [ 25.4] Thác nước có chiều cao nước rơi cách mặt sông lớn nhất châu Á là thác nước Jog trê ← BẰNG CHỨNG VÀNG
    2. [ 14.9] Trên tuyến sông, vào mùa nước trung thì tàu thuyền có thể khai thác thuận lợi, san
    3. [ 13.7] Hệ thống sông Vu Gia - Thu Bồn với phần lớn lưu vực nằm trong địa giới tỉnh được đ
    4. [ 13.3] Wonsan là thành phố duyên 

In [6]:
# Ô 6 — kiểm tiền đề của thí nghiệm trên chính dữ liệu vừa dựng. Khoảng 1 phút, CPU.
# Ba điều phải đúng, nếu không thì phép so cặp mất nghĩa và đừng tốn GPU chạy tiếp.
import sys

sys.path.insert(0, "src")
import numpy as np

from vihallulens.data.chunking import chunk_context
from vihallulens.data.loading import load_dataset

d = load_dataset("viwikifc_e08", "dev")
present = d[d["meta"].str["condition"] == "present"].sort_values("sample_id")
absent = d[d["meta"].str["condition"] == "absent"].sort_values("sample_id")
print(f"cặp                : {len(present):,} present / {len(absent):,} absent")
assert len(present) == len(absent), "cặp lệch nửa"

def cut(text):
    return [c.text for c in chunk_context(text, strategy="sentence", min_words=5)]

same, one = 0, 0
for a, b in zip(present["context"], absent["context"], strict=True):
    left, right = cut(a), cut(b)
    if len(left) == len(right):
        same += 1
        one += int(sum(x != y for x, y in zip(left, right, strict=True)) == 1)
print(f"hai vế cùng số đoạn: {same:,}/{len(present):,} ({same / len(present):.1%})")
print(f"khác đúng một đoạn : {one:,}/{same:,} ({one / same:.1%})")

pos = np.asarray([m["gold_position"] for m in present["meta"]])
k = np.median([len(cut(t)) for t in present["context"].head(200)])
print(f"vị trí đoạn vàng   : TB {pos.mean() / (k - 1):.3f}, ở đoạn đầu {(pos == 0).mean():.1%}")

assert same == len(present), "co cap lech so doan lot qua khau dung"
assert one == same, "co cap khac nhieu hon mot doan"
assert 0.35 < pos.mean() / (k - 1) < 0.65, "vi tri vang lech ve mot phia"
print("\nBa tien de deu dat.")

cặp                : 1,836 present / 1,836 absent
hai vế cùng số đoạn: 1,836/1,836 (100.0%)
khác đúng một đoạn : 1,836/1,836 (100.0%)
vị trí đoạn vàng   : TB 0.506, ở đoạn đầu 9.5%

Ba tien de deu dat.


## Trích đặc trưng

Một ô, khoảng **48 phút**: 3.672 dòng ở ~784 ms mỗi dòng. Chạy lại được.

**Đọc gì trong lúc chạy:** dòng `mẫu có bằng chứng` phải báo khoảng 1.836/3.672 — đúng một nửa,
vì chỉ nửa `present` mang bằng chứng. Nếu nó báo 0 thì cột `evidence` không tới được bộ trích và
phần định vị sẽ trống.

In [7]:
# Ô 7 — trích đặc trưng. Khoảng 48 phút.
!python scripts/extract_features.py --config configs/e08_extrinsic_viwikifc.yaml --split dev


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e08_extrinsic_viwikifc.yaml  (hash 16957fb67493)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : viwikifc_e08/dev, 3,672 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 1,836/3,672  → ghi thêm gold_rank cho E06
  đã có sẵn             : 0 mẫu trong viwikifc_e08_dev_16957fb67493.jsonl
  còn phải chạy         : 3,672 mẫu
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 2.77MB/s]
tokenizer_config.json: 7.30kB [00:00, 18.9MB/s]
vocab.json: 2.78MB [00:00, 69.3MB/s]
merges.txt: 1.67MB [00:00, 119MB/s]
tokenizer.json: 7.03MB [00:00, 124MB/s]
model.safetensors.index.json: 27.8kB [00:00, 78.7MB/s]
Fetching 4 files: 100%|█████████████

## Đo

Chạy CPU, vài giây.

In [8]:
# Ô 8 — phép kiểm can thiệp và phần định vị.
!python scripts/run_extrinsic.py --config configs/e08_extrinsic_viwikifc.yaml --split dev


E08 — BỎ BẰNG CHỨNG ĐI: PHÉP KIỂM CAN THIỆP
  cấu hình              : configs/e08_extrinsic_viwikifc.yaml  (trích 16957fb67493)
  bộ dữ liệu            : viwikifc_e08/dev, 3,672 dòng
  cặp đủ hai vế         : 1,836/1,836
  lưới lớp × đầu        : 27 × 28
  bị cắt ngữ cảnh       : 0

--------------------------------------------------------------------------------
BỎ BẰNG CHỨNG ĐI THÌ CHÚ Ý CÓ ĐỔI KHÔNG — 1,836 cặp
--------------------------------------------------------------------------------
  đặc trưng            có vàng  mất vàng       đổi  % cặp đúng hướng  cỡ ảnh hưởng
  chunk_entropy         0.7966    0.8176   +0.0156            75.4%      +0.7156  ✓
  chunk_max_share       0.3597    0.3392   -0.0139            72.4%      -0.6570  ✓
  chunk_gini            0.5064    0.4860   -0.0158            74.5%      -0.6855  ✓
  top1_top2_gap         0.1829    0.1611   -0.0129            71.2%      -0.6237  ✓
  chunk_drift           0.2081    0.2110   +0.0024            63.3%      +0.4009



In [9]:
# Ô 9 — kiểm toàn vẹn shard trước khi rời phiên. Vài giây, CPU.
import json
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

run = extraction_hash(load_config("configs/e08_extrinsic_viwikifc.yaml"))
path = Path("data/processed") / f"viwikifc_e08_dev_{run}.jsonl"
if not path.exists():
    raise SystemExit(f"THIEU {path.name} - chay lai o trich truoc khi roi phien.")
rows = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
ids = {r["sample_id"] for r in rows}
blocks = [k for k in rows[0] if k.startswith(("lookback_", "chunk_", "top1_"))]
gold = [r for r in rows if "gold_rank" in r]
ok = len(rows) == 3672 and len(ids) == len(rows) and len(blocks) == 7
print(f"  {path.name}")
print(f"  {len(rows):,}/3.672 dong, {len(ids):,} id, {len(blocks)} khoi dac trung")
print(f"  {len(gold):,} mau co gold_rank (nua 'present')")
print(f"  bi cat ngu canh: {sum(r['truncated'] for r in rows):,}")
print("\nShard hop le." if ok else "\nCO VAN DE - chay lai o trich truoc khi roi phien.")

  viwikifc_e08_dev_16957fb67493.jsonl
  3,672/3.672 dong, 3,672 id, 7 khoi dac trung
  1,836 mau co gold_rank (nua 'present')
  bi cat ngu canh: 0

Shard hop le.


In [10]:
# Ô 10 — lấy kết quả về.
import shutil
from pathlib import Path

for name in ("results/runs.jsonl", "data/interim/viwikifc_e08_dev.parquet"):
    shutil.copy(name, f"/kaggle/working/{Path(name).name}")
    print(Path(name).name)
for path in sorted(Path("data/processed").glob("viwikifc_e08_*.jsonl")):
    shutil.copy(path, f"/kaggle/working/{path.name}")
    print(f"{path.name}  {path.stat().st_size / 1024**2:,.0f} MB")

runs.jsonl
viwikifc_e08_dev.parquet
viwikifc_e08_dev_16957fb67493.jsonl  189 MB


## Sau khi chạy

Dán toàn bộ output của ô 8. Bảng cần lấy là `BỎ BẰNG CHỨNG ĐI THÌ CHÚ Ý CÓ ĐỔI KHÔNG` với đủ cả
ba cột `đổi`, `% cặp đúng hướng` và `cỡ ảnh hưởng`, cộng khối `ĐỊNH VỊ TRÊN NỬA CÓ VÀNG`.

**Đọc cột `đổi` cùng lúc với cột phần trăm.** Kiểm định theo cặp đo mức nhất quán của *hướng*,
không đo độ lớn — một dịch chuyển nhỏ tới mức vô nghĩa vẫn cho 100 % cặp đúng hướng nếu nó đều.
Có ca kiểm thử dựng sẵn đúng cái bẫy ấy.

Rồi Quick Save, tải notebook về, chép đè lên `notebooks/t27_bo_bang_chung_t4.ipynb`. **Đừng dùng
Save & Run All.**

Tải cả `runs.jsonl`, `viwikifc_e08_dev.parquet` và shard `viwikifc_e08_dev_*.jsonl` về — chấm
điểm sẽ chạy lại ở máy cá nhân theo quy tắc chốt ở T26, để mọi con số đem so đều cùng một máy.